# Post process data extracted using LLMs
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Eventually explode lists for geocoding?
4. Geocoding
5. Sanity checks

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
from matplotlib import pyplot as plt
from src.data import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *



In [2]:
#load data
model_name = "meta-llama/llama-4-scout-17b-16e-instruct"
nreports = 50
res_savename = f"llm_response_hazmain_nb_format_std_units_impact_{nreports}rep_test_{model_name.replace('/', '_')}.csv"
response_df = pd.read_csv(DATA_OUT_LLMS+res_savename)

In [3]:
#get rid of nans
response_df = response_df.dropna(subset=["nathaz_text"])

In [4]:
response_df

,impactType,impactValue,impactUnit,impactValueFlag,location,startYear,startMonth,startDay,endYear,endMonth,endDay,hazards,impactsAnnotation,appealCode,country_kw,reportDate,disasterType,nathaz_text,country
0,Affected People,45000.0,people,exact,"['Java', 'Nusa Tenggara', 'Bali', 'Kalimantan'...",2023.0,8.0,NaN,2024.0,6.0,NaN,['Drought'],['The drought operation aimed to address the i...,MDRID026,Indonesia,2025-05-14 00:00:00,Drought,['Page 1 24 Description of the Event Map of PM...,NaN
1,Agriculture,1000.0,microfarmers,approx,"['Malang', 'Pamekasan']",2024.0,NaN,NaN,NaN,NaN,NaN,['Drought'],['PMI supported 230 farmer households or 1150 ...,MDRID026,Indonesia,2025-05-14 00:00:00,Drought,['Page 1 24 Description of the Event Map of PM...,NaN
2,"Water, Sanitation, and Hygiene Infrastructure",7.0,water sources,exact,"['Jembrana', 'West Nusa Tenggara', 'East Nusa ...",NaN,NaN,NaN,NaN,NaN,NaN,['Drought'],['PMI rehabilitated damaged pipelines spanning...,MDRID026,Indonesia,2025-05-14 00:00:00,Drought,['Page 1 24 Description of the Event Map of PM...,NaN
3,Healthcare Infrastructure,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,['Drought'],['The local government had implemented mitigat...,MDRID026,Indonesia,2025-05-14 00:00:00,Drought,['Page 1 24 Description of the Event Map of PM...,NaN
4,Affected People,1000.0,people,exact,['Jijel'],2024.0,2.0,29.0,NaN,NaN,NaN,['Flood'],"['On February29th ,2024 , the Wilaya of Jijel ...",MDRDZ010,Algeria,2025-05-02 00:00:00,Flood,['DREF Final Report Algeria Flood 2024 Jijel f...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
343,Agriculture,4700000.0,livestock,exact,NaN,2024.0,3.0,NaN,NaN,NaN,NaN,['Extreme cold temperature'],['The extreme weather affected185937 herder ho...,MDRMN020,Mongolia,2025-02-27 00:00:00,Cold Wave,['SITUATION ANALYSIS Description of the crisis...,NaN
344,Agriculture,7.0,%,exact,NaN,2024.0,3.0,NaN,NaN,NaN,NaN,['Extreme cold temperature'],['The extreme weather affected185937 herder ho...,MDRMN020,Mongolia,2025-02-27 00:00:00,Cold Wave,['SITUATION ANALYSIS Description of the crisis...,NaN
345,Education Infrastructure,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,['Extreme cold temperature'],['This loss undermined food security by reduci...,MDRMN020,Mongolia,2025-02-27 00:00:00,Cold Wave,['SITUATION ANALYSIS Description of the crisis...,NaN
346,Agriculture,12.0,%,exact,NaN,2024.0,NaN,NaN,NaN,NaN,NaN,['Extreme cold temperature'],['The dramatic livestock losses resulted in a1...,MDRMN020,Mongolia,2025-02-27 00:00:00,Cold Wave,['SITUATION ANALYSIS Description of the crisis...,NaN


In [5]:
#convert numerical columns
num_cols = ["impactValue", "startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]
response_df_proc = cp.deepcopy(response_df)
response_df_proc = format_output(response_df_proc, num_cols=num_cols)


In [6]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(country_name_to_iso3)
response_df_proc["country_iso3_kw"] = response_df_proc["country_kw"].apply(country_name_to_iso3)

In [7]:
#format units
from spacy.lang.en import English
from spacy.lang.punctuation import TOKENIZER_PREFIXES, TOKENIZER_SUFFIXES, TOKENIZER_INFIXES
from spacy.lang.en import TOKENIZER_EXCEPTIONS
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex

## Sanity checks
1. ValueInText: ImpactValue should be in the original text
2. MaxPop: ImpactValue with “people” unit should be smaller than country’s population
3. OrigSentence: Annotation sentence should be present in the original text
4. UnknownImpType: Inferred impactType must be in the allowed list
5. UnknownHaz: Inferred hazardType must be in allowed list
6. Location partially undefined. 


In [8]:
response_df_proc.impactUnit.value_counts()

impactUnit
people                             139
houses                              25
homes                               17
kilometer**2                         9
families                             8
schools                              7
bridges                              5
livestock                            3
person                               3
hospital                             3
households                           3
farmers                              2
%                                    2
kilometer²                           2
buildings                            2
latrines                             2
children                             2
cans of harvested coffee beans       1
hospitals                            1
EUR                                  1
institutions                         1
health centers                       1
establishments                       1
water sources                        1
facilities                           1
health institu

In [9]:
response_df_proc.impactType.value_counts()

impactType
Affected People                                  63
Residential Buildings                            45
Agriculture                                      42
Displaced People                                 36
Human Deaths                                     30
Water, Sanitation, and Hygiene Infrastructure    29
Healthcare Infrastructure                        22
Transportation Infrastructure                    20
Education Infrastructure                         19
Injured People                                   15
Homeless People                                  11
Missing People                                    6
IT and Communication Infrastructure               5
Livestock                                         1
People with Disabilities                          1
Pregnant Women                                    1
Children                                          1
Name: count, dtype: int64

In [10]:
response_df_proc.hazards.value_counts()

hazards
['Flood']                                               185
['Earthquake']                                           38
['Drought']                                              19
['Tropical storm']                                       17
['Convective Storm']                                     15
['Flood', 'Convective Storm']                            12
['Wildfire', 'Extreme warm temperature']                  9
['Extreme cold temperature', 'Extra-tropical storm']      9
['Wildfire']                                              8
['Mass movement', 'Flood']                                8
['Volcanic activity', 'Earthquake']                       8
['Extreme warm temperature']                              7
['Extreme cold temperature']                              5
['Volcanic activity']                                     3
['Flood', 'Tropical storm']                               2
['Mass movement']                                         1
['Extreme warm temperature', 'Dr

In [23]:
#value in original text
def format_number(num):
    """
    Formats a number into a string, removing unnecessary trailing zeros
    and the decimal point if it's not needed.
    """
    if isinstance(num, float) and num.is_integer():
        # If the number is a float but represents an integer
        return str(int(num))
    return str(num).rstrip('0').rstrip('.') if '.' in str(num) else str(num)
def value_in_text(extract_df):
    extract_df["value_in_text"] = extract_df.apply(lambda x: format_number(x["impactValue"]) in "".join(x["nathaz_text"]), axis=1)
    return extract_df

response_df_proc = value_in_text(response_df_proc)

In [24]:
response_df_proc.value_in_text.value_counts()

value_in_text
True     305
False     42
Name: count, dtype: int64

In [27]:
response_df_proc[response_df_proc["value_in_text"] == False]

,impactType,impactValue,impactUnit,impactValueFlag,location,startYear,startMonth,startDay,endYear,endMonth,...,appealCode,country_kw,reportDate,disasterType,nathaz_text,country,country_iso3,country_iso3_kw,value_in_text,pop_cntry_check
37,Residential Buildings,1.007000e+03,houses,approx,"['Dohuk', 'Erbil']",2024.0,3.0,19.0,NaN,NaN,...,MDRIQ016,Iraq,2025-04-29 00:00:00,Pluvial/Flash Flood,['DREF Final Report Iraq PluvialFlash Flood 20...,NaN,None,IRQ,False,NaN
52,Affected People,3.800000e+06,people,exact,"['Puntland', 'Somaliland']",2025.0,4.0,NaN,NaN,6.0,...,MDRSO022,Somalia,2025-04-26 00:00:00,Drought,['DREF Operation SomaliaDrought Somalia RC obs...,NaN,None,SOM,False,True
62,Livestock,1.215100e+09,heads of livestock,exact,"['Cochabamba', 'Chuquisaca', 'La Paz', 'Potosi...",2025.0,4.0,1.0,NaN,NaN,...,MDRBO018,Bolivia,2025-04-26 00:00:00,Flood,['March 2025 Appeal MDRBO018 Country Bolivia H...,Bolivia,BOL,BOL,False,NaN
71,"Water, Sanitation, and Hygiene Infrastructure",NaN,NaN,NaN,['Cube Parish Quinind'],2025.0,3.0,13.0,NaN,NaN,...,MDREC027,Ecuador,2025-04-25 00:00:00,Flood,"['The situation in the area is critical , and ...",NaN,None,ECU,False,NaN
104,Affected People,2.800000e+06,people,exact,['23 ASAL counties'],NaN,4.0,NaN,NaN,6.0,...,MDRKE065,Kenya,2025-04-22 00:00:00,Drought,['DREF Operation Kenya Drought Crop Failure in...,NaN,None,KEN,False,True
114,Agriculture,NaN,NaN,exact,"['Morogoro', 'Mara']",2025.0,3.0,NaN,NaN,NaN,...,MDRTZ040,"Tanzania, United Republic Of",2025-04-10 00:00:00,Flood,['Photo courtesy TRCS Appeal MDRTZ040 Country ...,NaN,None,TZA,False,NaN
136,"Water, Sanitation, and Hygiene Infrastructure",NaN,NaN,NaN,['San Vicente de Chucur'],2024.0,11.0,8.0,NaN,NaN,...,MDRCO028,Colombia,2025-04-05 00:00:00,Flood,['Appeal MDRCO028 Total DREF Allocation CHF 45...,NaN,None,COL,False,NaN
137,Agriculture,NaN,NaN,NaN,['Norte de Santander'],2024.0,11.0,NaN,NaN,NaN,...,MDRCO028,Colombia,2025-04-05 00:00:00,Flood,['Appeal MDRCO028 Total DREF Allocation CHF 45...,NaN,None,COL,False,NaN
144,Healthcare Infrastructure,NaN,NaN,exact,['Sabah'],2025.0,3.0,8.0,NaN,NaN,...,MDRMY012,Malaysia,2025-03-28 00:00:00,Flood,['PhotoMRCS Appeal MDRMY012 Country Malaysia H...,NaN,None,MYS,False,NaN
145,Agriculture,NaN,NaN,exact,['Sabah'],2025.0,3.0,8.0,NaN,NaN,...,MDRMY012,Malaysia,2025-03-28 00:00:00,Flood,['PhotoMRCS Appeal MDRMY012 Country Malaysia H...,NaN,None,MYS,False,NaN


In [ ]:
#impacted people must be less than population
country_pop = pd.read_csv(DATA_PATH +"API_SP.POP.TOTL_DS2_en_csv_v2_131993/"+"API_SP.POP.TOTL_DS2_en_csv_v2_131993.csv",sep=',', header=2)
country_pop = country_pop.dropna(how="all",axis=1)
def pop_cntry_check(extracted_data, country_pop):
    def check_pop(x):
        year = str(pd.to_datetime(x["reportDate"]).year)
        year_check = year if year in country_pop[country_pop["Country Code"] == x["country_iso3_kw"]].columns else "2023"
        pop_year = country_pop[country_pop["Country Code"] == x["country_iso3_kw"]][year_check].values[0]
        return x["impactValue"] < pop_year if (x["impactUnit"] == "people" and not np.isnan(x["impactValue"]))  else np.nan
    extracted_data["pop_cntry_check"] = np.nan
    extracted_data["pop_cntry_check"] = extracted_data.apply(check_pop, axis=1)
    return extracted_data

response_df_proc = pop_cntry_check(response_df_proc, country_pop)

In [21]:
response_df_proc.pop_cntry_check.value_counts()

pop_cntry_check
True     138
False      1
Name: count, dtype: int64

In [22]:
response_df_proc[response_df_proc["pop_cntry_check"] == False]

,impactType,impactValue,impactUnit,impactValueFlag,location,startYear,startMonth,startDay,endYear,endMonth,...,appealCode,country_kw,reportDate,disasterType,nathaz_text,country,country_iso3,country_iso3_kw,value_in_text,pop_cntry_check
118,Displaced People,NaN,people,exact,"['Akmola', 'North Kazakhstan', 'West Kazakhsta...",2024.0,3.0,27.0,NaN,NaN,...,MDRKZ013,Kazakhstan,2025-04-09 00:00:00,Flood,['Page 1 13 Description of the Event Map of th...,NaN,None,KAZ,True,False
